# d-EPR null model

This notebook uses the same sample stays and POI exposure as the K-Visitation notebook, supplied in `data/sample/`. The synthetic stay order is used to estimate the exploration rate. The synthetic coordinates and POI counts provide the distances and density weights for the simulation. Each amenity category present contributes one POI.

`sample_places.csv` contains synthetic place attributes. Coordinates are generated around a fictional home at (385000, 6670000) metres in EPSG:3067, using bearings clockwise from grid north. Euclidean home distances in that CRS match `home_dist_km`; `lat` and `lng` are stored in WGS84 (EPSG:4326).

The simulator projects these WGS84 coordinates to EPSG:3067 and uses Euclidean distances in metres, with a 100 m minimum distance, for its inverse-square gravity weights.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

root = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / 'src' / 'null_model.py').is_file())
sys.path.insert(0, str(root / 'src'))

from k_visitation import calculate_k_visitation
from null_model import estimate_user_rho, simulate_grid_visits

## 1. Sample stays and exploration rate

The null model requires at least two locations. When the stay sequence is too short to estimate the exploration rate, the simulator uses its default of 0.6.

In [ ]:
sample_dir = root / 'data' / 'sample'
stays = pd.read_csv(sample_dir / 'sample_stays.csv', parse_dates=['start', 'end'])
sample_places = pd.read_csv(sample_dir / 'sample_places.csv', float_precision='round_trip')
seed = 42

if len(sample_places) < 2:
    raise ValueError('The d-EPR demo requires at least two locations.')

user_id = stays['user_id'].iloc[0]
model_stays = stays.rename(columns={'cell_id': 'stay_gid10'}).assign(visit_freq=1)
rho = estimate_user_rho(model_stays, user_id)
user_rho = {user_id: rho} if np.isfinite(rho) else {}
print(f'Exploration rate used: {user_rho.get(user_id, 0.6):.4f}')

## 2. Simulated visits

The simulation starts at the most frequently visited location. Exploration favours nearby locations with more POIs, and returns favour locations visited more often in the simulation.

In [ ]:
home_grid = 'toy_home'
user_homes = pd.DataFrame({'user_id': [user_id], 'home_gid9': [home_grid]})
city_grid = sample_places.rename(columns={'cell_id': 'h3_index'})
city_grid['log_poi'] = np.log1p(city_grid['poi_count'])
simulated = simulate_grid_visits(
    home_grid, model_stays, user_homes, city_grid, user_rho,
    max_steps=len(stays), seed=seed,
)
display(simulated)

## 3. Null alignment

The simulated visit counts are joined to the same location attributes and passed to K-Visitation to calculate $q_K^{null}$.

In [ ]:
locations = simulated.rename(columns={
    'stay_gid10': 'cell_id', 'visit_freq_synth': 'visit_freq',
}).merge(sample_places, on='cell_id', validate='many_to_one')
places_null, qk_null = calculate_k_visitation(locations, distance_col='home_dist_km')
display(qk_null)